In [1]:
%idle_timeout 60
%glue_version 4.0
%region us-east-1
%iam_role arn:aws:iam::772297441628:role/GlueInteractiveSessionRole

print("Glue Session Configuration Complete!")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Setting Glue version to: 4.0
Previous region: us-east-1
Setting new region to: us-east-1
Region is set to: us-east-1
Current iam_role is None
iam_role has been set to arn:aws:iam::772297441628:role/GlueInteractiveSessionRole.
Trying to create a Glue session for the kernel.
Session Type: etl
Idle Timeout: 60
Session ID: 2ac08d2a-f762-40f1-aeb0-10908ff20c0c
Applying the following default arguments:
--glue_kernel_version 1.0.9
--enable-glue-datacatalog true
Waiting for session 2ac08d2a-f762-40f1-aeb0-10908ff20c0c to get into ready status...
Session 2ac08d2a-f762-40f1-aeb0-10908ff20c0c has been creat

In [2]:
# Read all 4 tables from Glue Catalog
print("Reading tables from Glue Data Catalog...")

claims = spark.read.table("healthcare_db.claims")
patients = spark.read.table("healthcare_db.patients")
providers = spark.read.table("healthcare_db.providers")
icd_codes = spark.read.table("healthcare_db.icd_codes")

print(f"✓ Claims: {claims.count()} rows")
print(f"✓ Patients: {patients.count()} rows")
print(f"✓ Providers: {providers.count()} rows")
print(f"✓ ICD Codes: {icd_codes.count()} rows")

print("\nSample from claims table:")
claims.show(5)

Reading tables from Glue Data Catalog...
✓ Claims: 10001 rows
✓ Patients: 1001 rows
✓ Providers: 201 rows
✓ ICD Codes: 11 rows

Sample from claims table:
+---------+----------+-----------+--------------+------------+----------+------------+--------------+----------------------+------------+--------------+----------+
| claim_id|patient_id|provider_id|diagnosis_code|service_date|claim_date|claim_amount|allowed_amount|patient_responsibility|claim_status|procedure_code|claim_type|
+---------+----------+-----------+--------------+------------+----------+------------+--------------+----------------------+------------+--------------+----------+
| claim_id|patient_id|provider_id|diagnosis_code|service_date|claim_date|        null|          null|                  null|claim_status|procedure_code|claim_type|
|CLM000001|  PAT00834|   PRV00066|         C50.9|  2026-02-13|2026-02-15|     82442.6|      75577.84|              13013.93|    APPROVED|      CPT43779|OUTPATIENT|
|CLM000002|  PAT00021|   P

In [4]:
from pyspark.sql.functions import col, datediff, current_date

print("Cleaning and joining tables...")

# Filter out header rows using different syntax
claims_clean = claims.filter(claims.claim_id != "claim_id")
patients_clean = patients.filter(patients.patient_id != "patient_id")
providers_clean = providers.filter(providers.provider_id != "provider_id")
icd_codes_clean = icd_codes.filter(icd_codes.code != "code")

# Cast numeric columns to proper types
claims_clean = claims_clean \
    .withColumn("claim_amount", col("claim_amount").cast("double")) \
    .withColumn("allowed_amount", col("allowed_amount").cast("double")) \
    .withColumn("patient_responsibility", col("patient_responsibility").cast("double"))

patients_clean = patients_clean.withColumn("age", col("age").cast("int"))

# Join all tables
silver_df = claims_clean \
    .join(patients_clean, "patient_id", "left") \
    .join(providers_clean, "provider_id", "left") \
    .join(icd_codes_clean, claims_clean["diagnosis_code"] == icd_codes_clean["code"], "left")

print(f"✓ Joined table: {silver_df.count()} rows")
print("\nSample joined data:")
silver_df.select("claim_id", "patient_id", "age", "provider_name", "specialty", "claim_amount").show(5, False)

Cleaning and joining tables...
✓ Joined table: 10000 rows

Sample joined data:
+---------+----------+---+------------------------------+----------+------------+
|claim_id |patient_id|age|provider_name                 |specialty |claim_amount|
+---------+----------+---+------------------------------+----------+------------+
|CLM000002|PAT00021  |76 |Ramirez-Stephenson Pharmacy   |Pharmacy  |42714.74    |
|CLM000006|PAT00405  |41 |Dr. Brian Porter MD           |Psychiatry|97642.67    |
|CLM000004|PAT00629  |21 |Dr. Brittany Miller           |Neurology |103781.14   |
|CLM000001|PAT00834  |54 |East Karenshire Medical Center|Oncology  |82442.6     |
|CLM000003|PAT00179  |37 |Anderson Ltd Pharmacy         |Pharmacy  |716.04      |
+---------+----------+---+------------------------------+----------+------------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import when, datediff, to_date, floor

print("Adding calculated fields for Silver layer...")

# Add age_group
silver_enriched = silver_df.withColumn(
    "age_group",
    when(col("age") < 18, "0-17")
    .when((col("age") >= 18) & (col("age") < 40), "18-39")
    .when((col("age") >= 40) & (col("age") < 60), "40-59")
    .otherwise("60+")
)

# Add cost_category
silver_enriched = silver_enriched.withColumn(
    "cost_category",
    when(col("claim_amount") < 1000, "Low")
    .when((col("claim_amount") >= 1000) & (col("claim_amount") < 10000), "Medium")
    .when((col("claim_amount") >= 10000) & (col("claim_amount") < 50000), "High")
    .otherwise("Very High")
)

# Add processing_days (days between service and claim)
silver_enriched = silver_enriched.withColumn(
    "processing_days",
    datediff(to_date(col("claim_date")), to_date(col("service_date")))
)

print(f"✓ Silver enriched: {silver_enriched.count()} rows")
print("\nSample with calculated fields:")
silver_enriched.select("claim_id", "age", "age_group", "claim_amount", "cost_category", "processing_days").show(10, False)

Adding calculated fields for Silver layer...
✓ Silver enriched: 10000 rows

Sample with calculated fields:
+---------+---+---------+------------+-------------+---------------+
|claim_id |age|age_group|claim_amount|cost_category|processing_days|
+---------+---+---------+------------+-------------+---------------+
|CLM000002|76 |60+      |42714.74    |High         |6              |
|CLM000007|71 |60+      |661.07      |Low          |2              |
|CLM000011|55 |40-59    |81683.85    |Very High    |10             |
|CLM000006|41 |40-59    |97642.67    |Very High    |6              |
|CLM000004|21 |18-39    |103781.14   |Very High    |3              |
|CLM000001|54 |40-59    |82442.6     |Very High    |2              |
|CLM000010|74 |60+      |109224.06   |Very High    |8              |
|CLM000003|37 |18-39    |716.04      |Low          |6              |
|CLM000005|64 |60+      |31716.14    |High         |3              |
|CLM000008|79 |60+      |3476.99     |Medium       |9            

In [7]:
print("Writing Silver layer to S3...")

# Drop duplicate columns from the join
# Keep only one version of each duplicate column
silver_clean = silver_enriched.drop(providers_clean["state"]) \
    .drop(icd_codes_clean["name"]) \
    .drop(icd_codes_clean["category"]) \
    .drop(icd_codes_clean["severity"]) \
    .drop(icd_codes_clean["avg_cost"])

# Define S3 output path
silver_output_path = "s3://vedant-learning-processed-data/healthcare/silver/claims_enriched/"

# Write as Parquet (columnar format, compressed)
silver_clean.write \
    .mode("overwrite") \
    .parquet(silver_output_path)

print(f"✓ Silver layer written to: {silver_output_path}")
print(f"✓ Total rows written: {silver_clean.count()}")

# Verify by reading back
silver_verify = spark.read.parquet(silver_output_path)
print(f"✓ Verification - rows read back: {silver_verify.count()}")
print("\nColumns in Silver layer:")
print(silver_verify.columns)

Writing Silver layer to S3...
✓ Silver layer written to: s3://vedant-learning-processed-data/healthcare/silver/claims_enriched/
✓ Total rows written: 10000
✓ Verification - rows read back: 10000

Columns in Silver layer:
['provider_id', 'patient_id', 'claim_id', 'diagnosis_code', 'service_date', 'claim_date', 'claim_amount', 'allowed_amount', 'patient_responsibility', 'claim_status', 'procedure_code', 'claim_type', 'age', 'gender', 'state', 'zip_code', 'insurance_plan', 'chronic_conditions', 'member_since_date', 'provider_name', 'specialty', 'provider_type', 'city', 'hospital_name', 'years_experience', 'network_tier', 'code', 'age_group', 'cost_category', 'processing_days']


In [8]:
from pyspark.sql.functions import percent_rank
from pyspark.sql.window import Window

print("Creating KPI 1: High-Cost Claims Analysis...")

# Read Silver layer
silver = spark.read.parquet("s3://vedant-learning-processed-data/healthcare/silver/claims_enriched/")

# Create window to calculate percentile
window_spec = Window.orderBy(col("claim_amount").desc())

# Identify top 1% most expensive claims
high_cost_claims = silver.withColumn(
    "cost_percentile",
    percent_rank().over(window_spec)
).filter(col("cost_percentile") <= 0.01)

# Add analysis columns
high_cost_analysis = high_cost_claims.select(
    "claim_id",
    "patient_id",
    "age",
    "diagnosis_code",
    "provider_name",
    "specialty",
    "claim_amount",
    "allowed_amount",
    "patient_responsibility",
    "network_tier",
    "claim_status"
).orderBy(col("claim_amount").desc())

# Write to Gold layer
gold_path_1 = "s3://vedant-learning-processed-data/healthcare/gold/high_cost_claims/"
high_cost_analysis.write.mode("overwrite").parquet(gold_path_1)

print(f"✓ High-cost claims saved: {high_cost_analysis.count()} rows")
print("\nTop 5 most expensive claims:")
high_cost_analysis.show(5, False)

Creating KPI 1: High-Cost Claims Analysis...
✓ High-cost claims saved: 100 rows

Top 5 most expensive claims:
+---------+----------+---+--------------+--------------------------+---------------------------+------------+--------------+----------------------+------------+------------+
|claim_id |patient_id|age|diagnosis_code|provider_name             |specialty                  |claim_amount|allowed_amount|patient_responsibility|network_tier|claim_status|
+---------+----------+---+--------------+--------------------------+---------------------------+------------+--------------+----------------------+------------+------------+
|CLM006890|PAT00516  |48 |C50.9         |"Mitchell                 | Marquez and Wang Pharmacy"|110414.82   |99590.16      |11679.01              |20          |APPROVED    |
|CLM002085|PAT00279  |53 |C50.9         |Lake Nicole Medical Center|Primary Care               |110387.91   |100079.17     |18163.27              |IN_NETWORK  |APPROVED    |
|CLM006157|PAT00917 

In [9]:
from pyspark.sql.functions import count, avg, sum as spark_sum, rank
from pyspark.sql.window import Window

print("Creating KPI 2: Provider Performance Ranking...")

# Aggregate by provider
provider_performance = silver.groupBy(
    "provider_id",
    "provider_name", 
    "specialty",
    "network_tier"
).agg(
    count("claim_id").alias("total_claims"),
    avg("claim_amount").alias("avg_claim_amount"),
    spark_sum("claim_amount").alias("total_billed"),
    avg("patient_responsibility").alias("avg_patient_cost"),
    (count(when(col("claim_status") == "DENIED", 1)) / count("claim_id") * 100).alias("denial_rate_pct")
)

# Rank providers by total volume
window_rank = Window.orderBy(col("total_claims").desc())
provider_ranked = provider_performance.withColumn(
    "volume_rank",
    rank().over(window_rank)
)

# Write to Gold layer
gold_path_2 = "s3://vedant-learning-processed-data/healthcare/gold/provider_performance/"
provider_ranked.write.mode("overwrite").parquet(gold_path_2)

print(f"✓ Provider performance saved: {provider_ranked.count()} providers")
print("\nTop 10 providers by volume:")
provider_ranked.orderBy(col("total_claims").desc()).show(10, False)

Creating KPI 2: Provider Performance Ranking...
✓ Provider performance saved: 200 providers

Top 10 providers by volume:
+-----------+-------------------------+------------------------------+--------------+------------+------------------+------------------+------------------+------------------+-----------+
|provider_id|provider_name            |specialty                     |network_tier  |total_claims|avg_claim_amount  |total_billed      |avg_patient_cost  |denial_rate_pct   |volume_rank|
+-----------+-------------------------+------------------------------+--------------+------------+------------------+------------------+------------------+------------------+-----------+
|PRV00110   |Dr. Jasmine White        |Dermatology                   |IN_NETWORK    |66          |19181.49121212122 |1265978.4200000004|3459.3519696969693|6.0606060606060606|1          |
|PRV00178   |Hillburgh Medical Center |Dermatology                   |IN_NETWORK    |65          |22049.088307692306|1433190.739999

In [10]:
print("Creating KPI 3: Fraud Detection Indicators...")

# Identify suspicious patterns
fraud_indicators = silver.withColumn(
    "fraud_score",
    (
        # High claim amount for diagnosis
        when(col("claim_amount") > 100000, 3).otherwise(0) +
        # Very fast processing (same day = suspicious)
        when(col("processing_days") == 0, 2).otherwise(0) +
        # Out of network + very high cost
        when((col("network_tier") == "OUT_OF_NETWORK") & (col("claim_amount") > 50000), 2).otherwise(0) +
        # Patient responsibility > 50% (unusual)
        when(col("patient_responsibility") / col("allowed_amount") > 0.5, 1).otherwise(0)
    )
)

# Flag high-risk claims (fraud_score >= 3)
high_risk_claims = fraud_indicators.filter(col("fraud_score") >= 3).select(
    "claim_id",
    "patient_id",
    "provider_id",
    "provider_name",
    "diagnosis_code",
    "claim_amount",
    "processing_days",
    "network_tier",
    "fraud_score",
    "claim_status"
).orderBy(col("fraud_score").desc())

# Write to Gold layer
gold_path_3 = "s3://vedant-learning-processed-data/healthcare/gold/fraud_indicators/"
high_risk_claims.write.mode("overwrite").parquet(gold_path_3)

print(f"✓ Fraud indicators saved: {high_risk_claims.count()} high-risk claims")
print("\nTop 10 suspicious claims:")
high_risk_claims.show(10, False)

Creating KPI 3: Fraud Detection Indicators...
✓ Fraud indicators saved: 229 high-risk claims

Top 10 suspicious claims:
+---------+----------+-----------+---------------------------+--------------+------------+---------------+--------------+-----------+------------+
|claim_id |patient_id|provider_id|provider_name              |diagnosis_code|claim_amount|processing_days|network_tier  |fraud_score|claim_status|
+---------+----------+-----------+---------------------------+--------------+------------+---------------+--------------+-----------+------------+
|CLM002363|PAT00005  |PRV00140   |Sheppardfort Medical Center|C50.9         |107510.22   |4              |OUT_OF_NETWORK|5          |DENIED      |
|CLM000550|PAT00508  |PRV00047   |Dr. William Carter         |C50.9         |101095.5    |3              |OUT_OF_NETWORK|5          |DENIED      |
|CLM003068|PAT00368  |PRV00104   |Davidmouth Medical Center  |C50.9         |110170.87   |10             |OUT_OF_NETWORK|5          |DENIED      

In [11]:
print("Creating KPI 4: Diagnosis Cost Analysis...")

# Aggregate by diagnosis
diagnosis_analysis = silver.groupBy(
    "diagnosis_code"
).agg(
    count("claim_id").alias("total_claims"),
    avg("claim_amount").alias("avg_claim_cost"),
    spark_sum("claim_amount").alias("total_cost"),
    avg("processing_days").alias("avg_processing_days"),
    (count(when(col("claim_status") == "APPROVED", 1)) / count("claim_id") * 100).alias("approval_rate_pct")
).orderBy(col("total_cost").desc())

# Write to Gold layer
gold_path_4 = "s3://vedant-learning-processed-data/healthcare/gold/diagnosis_cost_analysis/"
diagnosis_analysis.write.mode("overwrite").parquet(gold_path_4)

print(f"✓ Diagnosis analysis saved: {diagnosis_analysis.count()} diagnoses")
print("\nTop diagnoses by total cost:")
diagnosis_analysis.show(10, False)

Creating KPI 4: Diagnosis Cost Analysis...
✓ Diagnosis analysis saved: 10 diagnoses

Top diagnoses by total cost:
+--------------+------------+------------------+--------------------+-------------------+-----------------+
|diagnosis_code|total_claims|avg_claim_cost    |total_cost          |avg_processing_days|approval_rate_pct|
+--------------+------------+------------------+--------------------+-------------------+-----------------+
|C50.9         |1041        |85437.38538904896 |8.894031818999997E7 |5.389048991354467  |84.14985590778097|
|N18.3         |1018        |44876.06238703337 |4.5683831509999976E7|5.514734774066798  |83.69351669941061|
|I25.10        |991         |35114.97427850655 |3.479893950999999E7 |5.4732593340060545 |84.6619576185671 |
|J44.9         |1001        |4963.355234765231 |4968318.589999996   |5.35964035964036   |85.61438561438561|
|M17.9         |992         |3994.613709677415 |3962656.7999999956  |5.465725806451613  |86.99596774193549|
|F41.1         |1047  

In [12]:
print("Creating KPI 5: Regional Cost Analysis...")

# Aggregate by state (patient location)
regional_analysis = silver.groupBy("state").agg(
    count("claim_id").alias("total_claims"),
    avg("claim_amount").alias("avg_claim_cost"),
    spark_sum("claim_amount").alias("total_cost"),
    avg("age").alias("avg_patient_age"),
    (count(when(col("network_tier") == "IN_NETWORK", 1)) / count("claim_id") * 100).alias("in_network_rate_pct"),
    (count(when(col("claim_status") == "DENIED", 1)) / count("claim_id") * 100).alias("denial_rate_pct")
).orderBy(col("total_cost").desc())

# Write to Gold layer
gold_path_5 = "s3://vedant-learning-processed-data/healthcare/gold/regional_analysis/"
regional_analysis.write.mode("overwrite").parquet(gold_path_5)

print(f"✓ Regional analysis saved: {regional_analysis.count()} states")
print("\nTop 15 states by total healthcare cost:")
regional_analysis.show(15, False)

Creating KPI 5: Regional Cost Analysis...
✓ Regional analysis saved: 59 states

Top 15 states by total healthcare cost:
+-----+------------+------------------+------------------+------------------+-------------------+------------------+
|state|total_claims|avg_claim_cost    |total_cost        |avg_patient_age   |in_network_rate_pct|denial_rate_pct   |
+-----+------------+------------------+------------------+------------------+-------------------+------------------+
|OH   |268         |22492.582052238795|6028011.989999997 |50.05223880597015 |75.3731343283582   |8.955223880597014 |
|PW   |269         |20368.014795539028|5478995.979999999 |49.29368029739777 |72.11895910780669  |9.293680297397769 |
|IA   |279         |17296.825663082443|4825814.360000002 |50.91756272401434 |71.68458781362007  |6.093189964157706 |
|SC   |234         |19343.40777777778 |4526357.42        |54.78632478632478 |73.50427350427351  |7.264957264957266 |
|OK   |227         |19916.860660792943|4521127.369999998 |50.

In [14]:
from pyspark.sql.functions import row_number

print("Creating KPI 6: Patient Risk Stratification...")

# Aggregate patient-level metrics
patient_risk = silver.groupBy(
    "patient_id",
    "age",
    "gender",
    "state",
    "insurance_plan",
    "chronic_conditions"
).agg(
    count("claim_id").alias("total_claims"),
    spark_sum("claim_amount").alias("lifetime_cost"),
    avg("claim_amount").alias("avg_claim_cost"),
    (count(when(col("claim_status") == "DENIED", 1)) / count("claim_id") * 100).alias("denial_rate")
)

# Calculate risk score
patient_risk_scored = patient_risk.withColumn(
    "risk_score",
    (
        # High utilization (many claims)
        when(col("total_claims") > 15, 3).when(col("total_claims") > 10, 2).when(col("total_claims") > 5, 1).otherwise(0) +
        # High cost
        when(col("lifetime_cost") > 200000, 3).when(col("lifetime_cost") > 100000, 2).when(col("lifetime_cost") > 50000, 1).otherwise(0) +
        # Chronic condition
        when(col("chronic_conditions") != "None", 2).otherwise(0) +
        # Age (elderly = higher risk)
        when(col("age") >= 65, 2).when(col("age") >= 50, 1).otherwise(0)
    )
).withColumn(
    "risk_tier",
    when(col("risk_score") >= 8, "Critical")
    .when(col("risk_score") >= 5, "High")
    .when(col("risk_score") >= 3, "Medium")
    .otherwise("Low")
)

# Write to Gold layer
gold_path_6 = "s3://vedant-learning-processed-data/healthcare/gold/patient_risk_stratification/"
patient_risk_scored.write.mode("overwrite").parquet(gold_path_6)

print(f"✓ Patient risk stratification saved: {patient_risk_scored.count()} patients")
print("\nRisk distribution:")
patient_risk_scored.groupBy("risk_tier").count().orderBy("count", ascending=False).show()
print("\nTop 10 highest-risk patients:")
patient_risk_scored.orderBy(col("risk_score").desc()).show(10, False)

Creating KPI 6: Patient Risk Stratification...
✓ Patient risk stratification saved: 1000 patients

Risk distribution:
+---------+-----+
|risk_tier|count|
+---------+-----+
|     High|  453|
|   Medium|  260|
| Critical|  193|
|      Low|   94|
+---------+-----+


Top 10 highest-risk patients:
+----------+---+------+-----+--------------+------------------+------------+------------------+------------------+------------------+----------+---------+
|patient_id|age|gender|state|insurance_plan|chronic_conditions|total_claims|lifetime_cost     |avg_claim_cost    |denial_rate       |risk_score|risk_tier|
+----------+---+------+-----+--------------+------------------+------------+------------------+------------------+------------------+----------+---------+
|PAT00412  |69 |M     |AL   |BRONZE        |Hypertension      |17          |293562.25         |17268.367647058825|5.88235294117647  |10        |Critical |
|PAT00761  |82 |F     |AR   |BRONZE        |Diabetes          |18          |309414.469